# Optimizing Raw Material Inventory Management of MSME Product Using XGBoost Regressor
## A Sales Prediction Approach

This notebook implements the data processing and modeling pipeline described in the paper:
**OPTIMIZING RAW MATERIAL INVENTORY MANAGEMENT OF MSME PRODUCT USING EXTREME GRADIENT BOOSTING (XGBOOST) REGRESSOR ALGORITHM: A SALES PREDICTION APPROACH**
by Muhammad Khusni Fikri, et al.

### Table of Contents
1. Import Libraries
2. Data Collection (Simulated)
3. Data Understanding & Preprocessing
    - Feature Reengineering
    - Data Integration
    - Data Cleansing
4. Data Modeling
    - Descriptive Analysis
    - Predictive Analysis (XGBoost)
5. Evaluation

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for reproducibility
np.random.seed(42)

### 2. Data Collection (Simulated)
Since the original dataset is not publicly available, we will simulate the data based on the characteristics described in the paper.
- **Sales Data**: Contains transactions from January 9, 2023, to June 30, 2023, with features like `Date`, `Mode` (Dine in, Online, Take Away), and `Sales Quantity`.
- **Weather Data**: Contains `Date`, `Temperature_Avg (°C)`, and `Relative Humidity_Avg (%)` for Semarang City.

In [ ]:
# Simulate Dates
dates = pd.date_range(start='2023-01-09', end='2023-06-30')

# 1. Simulate Sales Data
modes = ['Dine in', 'Online', 'Take Away']
sales_data_list = []

for date in dates:
    for mode in modes:
        if np.random.rand() > 0.1: # 90% chance to have sales for this mode on this day
            qty = np.random.randint(1, 50) if mode == 'Dine in' else np.random.randint(1, 20)
            sales_data_list.append({'Date': date, 'Mode': mode, 'Sales Quantity': qty})

df_sales_raw = pd.DataFrame(sales_data_list)
print("Raw Sales Data Sample:")
display(df_sales_raw.head())

# 2. Simulate Weather Data
weather_data = pd.DataFrame({
    'Date': dates,
    'Temperature_Avg (°C)': np.random.uniform(26.0, 32.0, len(dates)),
    'Relative Humidity_Avg (%)': np.random.uniform(70.0, 90.0, len(dates))
})
print("\nWeather Data Sample:")
display(weather_data.head())

### 3. Data Understanding & Preprocessing
#### 3.1 Feature Reengineering
Grouping the total `Sales Quantity` by `Date` and `Mode`, and converting it into a pivot table.

In [ ]:
# Group by Date and Mode, then calculate the sum of Sales Quantity
df_sum = df_sales_raw.groupby([pd.Grouper(key='Date'), pd.Grouper(key='Mode')])['Sales Quantity'].sum().reset_index()
data_sum_daily = pd.DataFrame(df_sum)

# Create a pivot table to have Modes as columns
data_transpose = data_sum_daily.pivot(index='Date', columns='Mode', values='Sales Quantity')
print("Feature Reengineered Data:")
display(data_transpose.head())

#### 3.2 Data Integration
Merging the sales data with the weather data based on the `Date`.

In [ ]:
# Merge sales and weather data
data_final = pd.merge(data_transpose, weather_data, on='Date', how='inner')
print("Integrated Data Sample:")
display(data_final.head())

#### 3.3 Data Cleansing
Filling missing values (NaN) with 0, as some days might not have sales for a specific mode.

In [ ]:
# Fill missing values with 0
data_final.fillna(0, inplace=True)
print("Cleaned Data Sample:")
display(data_final.head())

### 4. Data Modeling
#### 4.1 Descriptive Analysis
Correlation analysis to understand the relationship between weather features and sales.

In [ ]:
# Calculate correlation matrix
corr = data_final[['Dine in', 'Online', 'Take Away', 'Temperature_Avg (°C)', 'Relative Humidity_Avg (%)']].corr()

# Plot correlation matrix using Seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation Diagram")
plt.show()

#### 4.2 Predictive Analysis (XGBoost)
Training an XGBoost Regressor model to predict sales based on average temperature and average humidity.

In [ ]:
# Features (X) and Targets (y)
X = data_final[['Temperature_Avg (°C)', 'Relative Humidity_Avg (%)']]
y = data_final[['Dine in', 'Online', 'Take Away']]

# Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize XGBoost Regressor
model = xgb.XGBRegressor(
    max_depth=3,
    n_estimators=100,
    learning_rate=0.1,
    objective='reg:squarederror'
)

# Train the model
model.fit(X_train, y_train)
print("Model training completed.")

### 5. Evaluation
Evaluating the model using Mean Absolute Error (MAE), Mean Squared Error (MSE), and Root Mean Squared Error (RMSE).

In [ ]:
# Predict on test data
predictions = model.predict(X_test)

# Calculate metrics
mse = mean_squared_error(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"Mean Squared Error: {mse:.14f}")
print(f"Mean Absolute Error: {mae:.14f}")
print(f"Root Mean Squared Error: {rmse:.14f}")

# Visualizing the predictions vs actuals for the first target (Dine in) as an example
plt.figure(figsize=(10, 5))
plt.plot(y_test['Dine in'].values, label='Actual (Dine in)', marker='o')
plt.plot(predictions[:, 0], label='Predicted (Dine in)', marker='x')
plt.title('Actual vs Predicted Sales (Dine in)')
plt.legend()
plt.show()